# Obesify — Entrega 2: EDA e Pré-processamento

**Projeto:** Classificação do estado nutricional  
**Dataset:** Estimation of Obesity Levels Based on Eating Habits and Physical Condition — UCI (ID 544)  
**Alvo:** `NObeyesdad`  
**Tipo:** classificação multiclasse (7 classes)

## Objetivos da entrega
1. Responder às cinco perguntas da EDA.
2. Documentar as decisões tomadas para cada variável.
3. Criar `limpar(df)` para operações linha a linha.
4. Construir o pré-processador com `ColumnTransformer`.
5. Verificar ausência de NaN após a transformação e comparar um modelo simples com o baseline.

## 0. Instalação e importações

In [ ]:
# %pip install -q ucimlrepo pandas numpy matplotlib seaborn scikit-learn

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

RANDOM_STATE = 42
TARGET = "NObeyesdad"

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

## 1. Carregamento dos dados e primeiro contato

O dataset UCI possui 16 variáveis preditoras e uma variável-alvo.

In [ ]:
obesity = fetch_ucirepo(id=544)

X_uci = obesity.data.features.copy()
y_uci = obesity.data.targets.copy()

df_original = pd.concat([X_uci, y_uci], axis=1)
if TARGET not in df_original.columns:
    df_original = df_original.rename(columns={df_original.columns[-1]: TARGET})

print("Dimensões:", df_original.shape)
display(df_original.head())
display(df_original.info())
print("\\nValores ausentes por coluna:")
display(df_original.isna().sum().to_frame("valores_ausentes"))
print("\\nDuplicatas exatas:", df_original.duplicated().sum())

## 2. Limpeza linha a linha

Não serão removidas observações apenas por parecerem extremas em um boxplot.

Duplicatas exatas são contabilizadas e removidas antes da divisão, conforme a orientação da Entrega 2. A quantidade removida será exibida para transparência.

In [ ]:
def limpar(df):
    dados = df.copy()

    # Padronização
    colunas_texto = dados.select_dtypes(include=["object", "category"]).columns
    for coluna in colunas_texto:
        dados[coluna] = dados[coluna].astype("string").str.strip()

    # Conversão numérica explícita
    colunas_numericas = ["Age", "Height", "Weight", "FCVC", "NCP", "CH2O", "FAF", "TUE"]
    for coluna in colunas_numericas:
        dados[coluna] = pd.to_numeric(dados[coluna], errors="coerce")

    # Valores impossíveis são tratados como ausentes.
    if "Age" in dados:
        dados.loc[(dados["Age"] <= 0) | (dados["Age"] > 120), "Age"] = np.nan
    for coluna in ["Height", "Weight"]:
        if coluna in dados:
            dados.loc[dados[coluna] <= 0, coluna] = np.nan

    return dados

df_limpo = limpar(df_original)
duplicatas = int(df_limpo.duplicated().sum())
print(f"Duplicatas exatas identificadas: {duplicatas}")

df_limpo = df_limpo.drop_duplicates().reset_index(drop=True)
print("Dimensões após limpeza:", df_limpo.shape)
print("Duplicatas restantes:", df_limpo.duplicated().sum())

## 3. Separação entre treino e teste

In [ ]:
X = df_limpo.drop(columns=[TARGET])
y = df_limpo[TARGET]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Treino:", X_treino.shape, y_treino.shape)
print("Teste reservado:", X_teste.shape, y_teste.shape)
print("Proporções das classes no treino:")
display(y_treino.value_counts(normalize=True).mul(100).round(2).rename("percentual").to_frame())

# 4. EDA com propósito — cinco perguntas

## Pergunta 1 — Como o alvo se distribui?

A distribuição do alvo mostra a frequência de cada uma das sete classes. A proporção deve ser registrada porque influencia a escolha da métrica e a interpretação do baseline. A divisão foi feita com `stratify=y`.

In [ ]:
contagem_alvo = y_treino.value_counts().sort_values(ascending=False)
percentual_alvo = y_treino.value_counts(normalize=True).reindex(contagem_alvo.index) * 100

plt.figure(figsize=(12, 5))
ax = sns.barplot(x=contagem_alvo.index, y=contagem_alvo.values)
ax.set_title("Distribuição das classes — conjunto de treino")
ax.set_xlabel("Classe nutricional")
ax.set_ylabel("Quantidade de registros")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

tabela_alvo = pd.DataFrame({"quantidade": contagem_alvo, "percentual": percentual_alvo.round(2)})
display(tabela_alvo)

print("Observação: a classe mais frequente representa "
      f"{percentual_alvo.max():.2f}% do treino; a menos frequente representa "
      f"{percentual_alvo.min():.2f}%.")
print("Decisão: manter a divisão estratificada e utilizar F1-macro como referência principal, "
      "pois considera o desempenho em todas as classes. O tratamento de eventual desbalanceamento "
      "(por exemplo, class_weight) será avaliado na Entrega 3, sem reamostrar o dataset inteiro.")

## Pergunta 2 — Onde faltam dados?

A análise verifica ausências reais e a proporção de valores ausentes por coluna no treino. A UCI informa que o dataset não possui valores faltantes, mas a conferência é repetida após o carregamento e a limpeza.

In [ ]:
ausentes = X_treino.isna().sum()
percentual_ausente = X_treino.isna().mean() * 100
tabela_ausentes = pd.DataFrame({
    "quantidade_ausente": ausentes,
    "percentual_ausente": percentual_ausente.round(2)
}).sort_values("percentual_ausente", ascending=False)
display(tabela_ausentes)

plt.figure(figsize=(11, 4))
sns.barplot(x=tabela_ausentes.index, y=tabela_ausentes["percentual_ausente"])
plt.title("Percentual de valores ausentes por variável — treino")
plt.ylabel("% de valores ausentes")
plt.xlabel("Variável")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

total_ausentes = int(ausentes.sum())
print(f"Observação: foram encontrados {total_ausentes} valores ausentes nas variáveis de entrada do treino.")
print("Decisão: manter SimpleImputer no pipeline como proteção para dados futuros/API. "
      "Usar mediana nas variáveis numéricas e moda nas categóricas. "
      "Não utilizar dropna(), para não descartar registros.")

## Pergunta 3 — Como cada variável se distribui?

Histogramas e boxplots ajudam a identificar assimetria, concentração e valores extremos. Um ponto fora do boxplot não é automaticamente um erro: valores plausíveis serão mantidos. O conjunto de teste não participa desta análise.

In [ ]:
num_cols = ["Age", "Height", "Weight", "FCVC", "NCP", "CH2O", "FAF", "TUE"]
cat_cols = [c for c in X_treino.columns if c not in num_cols]

display(X_treino[num_cols].describe().T)

X_treino[num_cols].hist(figsize=(14, 10), bins=25, edgecolor="black")
plt.suptitle("Distribuição das variáveis numéricas — treino", y=1.02)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 4, figsize=(15, 8))
for coluna, ax in zip(num_cols, axes.flat):
    sns.boxplot(y=X_treino[coluna], ax=ax)
    ax.set_title(coluna)
plt.tight_layout()
plt.show()

for coluna in cat_cols:
    print(f"\n{coluna}")
    display(X_treino[coluna].value_counts(dropna=False).to_frame("quantidade"))

print("Observação: inspecione os histogramas e boxplots acima para identificar assimetria "
      "e valores extremos. As escalas numéricas são diferentes.")
print("Decisão: aplicar mediana + StandardScaler às variáveis numéricas; moda + OneHotEncoder "
      "às categóricas. Não excluir valores extremos apenas por critério estatístico; só corrigir "
      "valores fisicamente inválidos identificados por regra de domínio na função limpar(df).")

## Pergunta 4 — O que se relaciona com o alvo?

São comparadas as distribuições de variáveis numéricas entre classes e as frequências das categorias. Essas relações são exploratórias e não demonstram causalidade. `Weight` e `Height` são especialmente importantes, pois a classificação nutricional possui relação antropométrica direta.

In [ ]:
for coluna in ["Age", "Height", "Weight", "FCVC", "FAF"]:
    plt.figure(figsize=(12, 5))
    sns.boxplot(data=pd.concat([X_treino[[coluna]], y_treino.rename(TARGET)], axis=1),
                x=TARGET, y=coluna)
    plt.title(f"{coluna} por classe nutricional — treino")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()

# Proporções por classe para variáveis categóricas selecionadas
for coluna in ["Gender", "family_history_with_overweight", "FAVC", "CAEC", "MTRANS"]:
    tabela = pd.crosstab(X_treino[coluna], y_treino, normalize="columns")
    tabela.plot(kind="bar", figsize=(12, 5))
    plt.title(f"{coluna}: proporção das categorias dentro de cada classe")
    plt.ylabel("Proporção")
    plt.xlabel(coluna)
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()

print("Observação: compare as medianas e amplitudes por classe nos boxplots e as proporções "
      "nas tabelas/gráficos categóricos.")
print("Decisão: manter as variáveis disponíveis que podem ser informadas no formulário. "
      "Não remover Height ou Weight por sua forte associação com o alvo, mas documentar que "
      "isso pode tornar a classificação mais fácil e não equivale a diagnóstico clínico. "
      "Associações observadas não devem ser descritas como causas.")

## Pergunta 5 — Existem variáveis numéricas redundantes?

A matriz de correlação de Pearson é calculada somente com as variáveis numéricas do treino. Correlação elevada (|r| > 0,90) é um alerta para investigação, não uma ordem automática de remoção.

In [ ]:
correlacao = X_treino[num_cols].corr(numeric_only=True)

plt.figure(figsize=(10, 7))
sns.heatmap(correlacao, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlação entre variáveis numéricas — treino")
plt.tight_layout()
plt.show()

pares_redundantes = []
for i, col_a in enumerate(correlacao.columns):
    for col_b in correlacao.columns[i+1:]:
        valor = correlacao.loc[col_a, col_b]
        if abs(valor) > 0.90:
            pares_redundantes.append((col_a, col_b, valor))

if pares_redundantes:
    print("Pares com |r| > 0,90:")
    for a, b, r in pares_redundantes:
        print(f"{a} x {b}: r={r:.3f}")
else:
    print("Não foram encontrados pares com |r| > 0,90 no treino.")

print("Decisão: não remover variáveis automaticamente nesta etapa. "
      "Se houver par com |r| > 0,90, avaliar relevância para o formulário e para o domínio "
      "antes de decidir. A decisão final deve ser registrada na tabela do README.")

# 5. Tabela de decisões por variável

A tabela abaixo documenta o tratamento planejado. Todas as 16 variáveis de entrada são mantidas nesta versão inicial; a variável-alvo não entra no pré-processamento.

In [ ]:
decisoes = pd.DataFrame([
    ["Gender", "Categórica nominal", "Sem ausências informadas pela UCI", "Manter; OneHotEncoder", "Pipeline"],
    ["Age", "Numérica", "Verificar distribuição e valores inválidos", "Idade <= 0 ou > 120 vira NaN; mediana + escala", "limpar(df) + Pipeline"],
    ["Height", "Numérica", "Verificar extremos e relação com alvo", "Valores <= 0 viram NaN; mediana + escala; manter valores plausíveis", "limpar(df) + Pipeline"],
    ["Weight", "Numérica", "Verificar extremos e relação com alvo", "Valores <= 0 viram NaN; mediana + escala; manter valores plausíveis", "limpar(df) + Pipeline"],
    ["family_history_with_overweight", "Binária nominal", "Resposta yes/no", "Manter; moda + OneHotEncoder", "Pipeline"],
    ["FAVC", "Binária nominal", "Resposta yes/no", "Manter; moda + OneHotEncoder", "Pipeline"],
    ["FCVC", "Numérica ordinal", "Escala de frequência de vegetais", "Manter como numérica nesta versão; mediana + escala", "Pipeline"],
    ["NCP", "Numérica", "Número de refeições principais", "Manter; mediana + escala", "Pipeline"],
    ["CAEC", "Categórica nominal", "Frequência de alimentação entre refeições", "Manter; moda + OneHotEncoder", "Pipeline"],
    ["SMOKE", "Binária nominal", "Resposta yes/no", "Manter; moda + OneHotEncoder", "Pipeline"],
    ["CH2O", "Numérica ordinal", "Escala de consumo de água", "Manter como numérica nesta versão; mediana + escala", "Pipeline"],
    ["SCC", "Binária nominal", "Monitoramento de calorias yes/no", "Manter; moda + OneHotEncoder", "Pipeline"],
    ["FAF", "Numérica ordinal", "Frequência de atividade física", "Manter como numérica nesta versão; mediana + escala", "Pipeline"],
    ["TUE", "Numérica ordinal", "Tempo de uso de tecnologia", "Manter como numérica nesta versão; mediana + escala", "Pipeline"],
    ["CALC", "Categórica nominal", "Frequência de consumo de álcool", "Manter; moda + OneHotEncoder", "Pipeline"],
    ["MTRANS", "Categórica nominal", "Meio de transporte", "Manter; moda + OneHotEncoder", "Pipeline"],
    ["NObeyesdad", "Alvo", "Sete classes de classificação", "Separar como y; nunca usar como entrada", "Fora do pré-processamento"]
], columns=["Coluna", "Tipo", "Achado/checagem", "Decisão", "Onde"])
display(decisoes)

# 6. Pré-processamento com ColumnTransformer

Variáveis numéricas recebem imputação pela mediana e padronização. Variáveis categóricas recebem imputação pela moda e One-Hot Encoding com `handle_unknown='ignore'`, evitando erro quando a API receber uma categoria não vista no treino.

O `ColumnTransformer` será inserido em um `Pipeline`. Assim, durante a validação cruzada, cada dobra aprende imputação, escala e categorias somente a partir da parte de treino daquela dobra.

In [ ]:
# Listas de colunas derivadas exclusivamente dos nomes/tipos do conjunto de treino.
num_cols = X_treino.select_dtypes(include=["number"]).columns.tolist()
cat_cols = [c for c in X_treino.columns if c not in num_cols]

print("Numéricas:", num_cols)
print("Categóricas:", cat_cols)

pipeline_num = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

pipeline_cat = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessador = ColumnTransformer(
    transformers=[
        ("num", pipeline_num, num_cols),
        ("cat", pipeline_cat, cat_cols)
    ],
    remainder="drop"
)

## 6.1 Teste do pré-processador

O `fit_transform` abaixo é usado somente em `X_treino` para verificar se a transformação funciona. Não aplicar `fit` ou `fit_transform` em `X_teste`.

In [ ]:
X_treino_transformado = preprocessador.fit_transform(X_treino)

matriz = X_treino_transformado.toarray() if hasattr(X_treino_transformado, "toarray") else np.asarray(X_treino_transformado)

print("Formato antes:", X_treino.shape)
print("Formato depois:", matriz.shape)
print("Quantidade de NaN após transformação:", int(np.isnan(matriz).sum()))
print("Quantidade de linhas antes/depois:", X_treino.shape[0], matriz.shape[0])

assert matriz.shape[0] == X_treino.shape[0], "A quantidade de linhas foi alterada!"
assert np.isnan(matriz).sum() == 0, "Ainda existem NaN após o pré-processamento!"
print("Pré-processador aprovado no teste de sanidade.")

# 7. Teste de sanidade com modelo simples e baseline

A comparação usa F1-macro, definida na Entrega 1, para considerar as sete classes. O baseline `DummyClassifier(strategy='most_frequent')` e a regressão logística são avaliados nas mesmas cinco dobras estratificadas do conjunto de treino.

O pré-processador está dentro do pipeline avaliado; portanto, é reajustado em cada dobra, evitando vazamento de dados.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")

modelo_simples = Pipeline(steps=[
    ("preprocessador", preprocessador),
    ("modelo", LogisticRegression(max_iter=3000, class_weight=None))
])

scores_baseline = cross_val_score(
    baseline, X_treino, y_treino,
    cv=5, scoring="f1_macro", n_jobs=-1
)
scores_modelo = cross_val_score(
    modelo_simples, X_treino, y_treino,
    cv=5, scoring="f1_macro", n_jobs=-1
)

resultado_cv = pd.DataFrame({
    "modelo": ["DummyClassifier (baseline)", "Regressão Logística"],
    "F1_macro_medio": [scores_baseline.mean(), scores_modelo.mean()],
    "desvio_padrao": [scores_baseline.std(), scores_modelo.std()]
})
display(resultado_cv)

if scores_modelo.mean() > scores_baseline.mean():
    print("Resultado: o modelo simples superou o baseline em F1-macro.")
else:
    print("Resultado: o modelo simples ainda não superou o baseline. "
          "Investigar dados, representação das variáveis e modelos na Entrega 3.")

print("Baseline F1-macro:", round(scores_baseline.mean(), 4))
print("Regressão Logística F1-macro:", round(scores_modelo.mean(), 4))

# 8. Conclusão da Entrega 2

- **Distribuição do alvo:** consultar a tabela e o gráfico da Pergunta 1; registrar se há desbalanceamento relevante.
- **Dados ausentes:** registrar a quantidade observada na Pergunta 2.
- **Distribuições e outliers:** manter valores plausíveis; corrigir somente erros de domínio identificáveis.
- **Relação com o alvo:** `Weight` e `Height` são variáveis antropométricas relevantes, mas a relação com a classificação deve ser tratada como limitação do problema.
- **Redundância:** registrar os pares com |r| > 0,90, se existirem, e justificar qualquer remoção.
- **Pré-processamento:** `ColumnTransformer` com imputação, escala e One-Hot Encoding, dentro de um Pipeline.
- **Sanidade:** confirmar zero NaN, quantidade de linhas preservada e comparação com o baseline.